# ⚽ FIFA Player Position Classification (v1.2) - Google Colab Runner

This notebook trains and evaluates the **v1.2** multi-season FIFA player position classification pipeline:
- **Training Set (FIFA 15–20)**: ~56,066 single-position players across 6 historical seasons.
- **Held-Out Test Set (FIFA 21)**: ~10,068 unseen players from the future season.
- **Target Classes (10 Positions)**: `CAM`, `CB`, `CDM`, `CM`, `GK`, `LB`, `LW`, `RB`, `RW`, `ST`.
- **Algorithms**: Logistic Regression, Random Forest, K-Nearest Neighbors (KNN), Support Vector Machine (SVM), and Soft-Voting Ensemble.
- **Persistence**: All models (`models/v1.2/`), figures (`figures/v1.2/`), and configs are saved directly into your **Google Drive** in real time!


## 1. Mount Google Drive & Set Working Directory
Connect to Google Drive and navigate to your `fifa-classification` project folder.


In [ ]:
from google.colab import drive
import os
import sys

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Path to project directory in Google Drive
PROJECT_DIR = '/content/drive/MyDrive/fifa-classification'

if not os.path.exists(PROJECT_DIR):
    print(f"Warning: '{PROJECT_DIR}' not found at exact path. Checking MyDrive contents...")
    available_dirs = [d for d in os.listdir('/content/drive/MyDrive') if not d.startswith('.')]
    print("Folders found in your Google Drive:", available_dirs)
    raise FileNotFoundError(f"Could not locate '{PROJECT_DIR}'. Please check the folder name in MyDrive.")

os.chdir(PROJECT_DIR)
print(f"✔ Current working directory set to: {os.getcwd()}")
print("\nDirectory contents:")
for item in sorted(os.listdir('.')):
    print(f"  - {item}")


## 2. Install Dependencies & Check Project Structure
Install required Python packages and verify all 7 raw CSV files and model directories are ready.


In [ ]:
# Install dependencies
!pip install --quiet pandas numpy scikit-learn matplotlib seaborn joblib

import glob

print("=== Checking Raw Data Files ===")
raw_files = sorted(glob.glob('data/raw/players_*.csv'))
print(f"Found {len(raw_files)}/7 raw CSV files:")
for f in raw_files:
    print(f"  ✔ {f}")

if len(raw_files) < 7:
    print("\n⚠️ WARNING: Not all players_*.csv files were found in data/raw/!")
    print("Please ensure players_15.csv through players_21.csv are uploaded to data/raw/.")
else:
    print("\n✔ All 7 FIFA season datasets (FIFA 15 to 21) are present and ready!")

print("\n=== Checking Pre-Existing Models for Version Comparison ===")
v1_models = glob.glob('models/v1/*.pkl')
v1_1_models = glob.glob('models/v1.1/*.pkl')
print(f"v1 Models:   {len(v1_models)} found in models/v1/")
print(f"v1.1 Models: {len(v1_1_models)} found in models/v1.1/")
if len(v1_models) > 0 and len(v1_1_models) > 0:
    print("✔ Models for v1 and v1.1 are present! Multi-version comparison (v1 vs v1.1 vs v1.2) will run automatically.")
else:
    print("ℹ️ Note: If v1 or v1.1 models are missing, version comparison will skip those versions.")


## 3. Data Preprocessing
Preprocess datasets for:
1. **v1 & v1.1** (ensures `test_data.csv` is present for multi-version comparison against v1.2).
2. **v1.2** (merges FIFA 15–20 for training, and prepares unseen FIFA 21 for testing).


In [ ]:
%%time
import os

# 1. Ensure test_data.csv exists for v1 and v1.1 for 3-way version comparison
if not os.path.exists('data/processed/v1/test_data.csv'):
    print("Generating processed test data for v1...")
    !python scripts/02-pre-process-data.py --config configs/v1.json

if not os.path.exists('data/processed/v1.1/test_data.csv'):
    print("\nGenerating processed test data for v1.1...")
    !python scripts/02-pre-process-data.py --config configs/v1.1.json

# 2. Preprocess v1.2 multi-season data
print("\n=== Preprocessing v1.2 Multi-Season Data ===")
!python scripts/02-pre-process-data.py --config configs/v1.2.json


## 4. Model Training & Hyperparameter Tuning
Each model has its own dedicated cell with a `%%time` timer.
- You can run them one by one in any order.
- Each script performs 5-fold cross-validation `GridSearchCV`, writes winning parameters to `configs/v1.2.json`, saves the fitted `.pkl` model to `models/v1.2/`, and writes confusion matrices to `figures/v1.2/`.
- All outputs are saved directly to your **Google Drive** immediately!


### Model 1: Logistic Regression
Multinomial classification with L2 penalty and standard scaling. (Typically finishes in ~2–5 minutes).


In [ ]:
%%time
print("=== Starting Logistic Regression Tuning & Training (v1.2) ===")
!python scripts/05-logistic-regression.py --config configs/v1.2.json --tune


### Model 2: Random Forest
Multi-tree ensemble with 500, 700, and 900 estimators, max depths, and leaf regularization. (Typically finishes in ~30–60 minutes).


In [ ]:
%%time
print("=== Starting Random Forest Tuning & Training (v1.2) ===")
!python scripts/06-random-forest.py --config configs/v1.2.json --tune


### Model 3: K-Nearest Neighbors (KNN)
Grid search over number of neighbors (k=1..39), distance metrics (Euclidean vs Manhattan), and weights.


In [ ]:
%%time
print("=== Starting K-Nearest Neighbors Tuning & Training (v1.2) ===")
!python scripts/03-knn.py --config configs/v1.2.json --tune


### Model 4: Support Vector Machine (SVM)
Non-linear RBF kernel classifier with probability estimation enabled for ensemble soft voting.


In [ ]:
%%time
print("=== Starting Support Vector Machine Tuning & Training (v1.2) ===")
!python scripts/04-svm.py --config configs/v1.2.json --tune


### Model 5: Soft-Voting Ensemble
Loads the tuned parameters of KNN, Logistic Regression, Random Forest, and SVM from `configs/v1.2.json` and tunes voting weights across the ensemble.


In [ ]:
%%time
print("=== Starting Soft-Voting Ensemble Tuning & Training (v1.2) ===")
!python scripts/07-ensemble.py --config configs/v1.2.json --tune


## 5. Comprehensive Multi-Model & Multi-Version Evaluation
Evaluates all trained v1.2 models on unseen FIFA 21 players and compares performance across **v1, v1.1, and v1.2**!
- Generates side-by-side performance metrics table.
- Generates `figures/v1.2/model_comparison_v1.2.png`.
- Generates `figures/v1.2/per_class_f1_heatmap_v1.2.png`.
- Generates `figures/v1.2/feature_importance_rf_v1.2.png`.
- Generates `figures/v1.2/height_weight_by_position_v1.2.png`.
- Generates `figures/v1.2/version_comparison_all.png` (v1 vs v1.1 vs v1.2)!


In [ ]:
%%time
print("=== Running Comprehensive Evaluation for v1.2 & Cross-Version Comparison ===")
!python scripts/08-evaluate-all.py --config configs/v1.2.json


## 6. Display All Generated Figures Inline
Display all charts, heatmaps, and confusion matrices directly inside this notebook. All files are permanently saved in `figures/v1.2/` on your Google Drive.


In [ ]:
import glob
from IPython.display import Image, display

print("===================================================")
print("         FIFA v1.2 EVALUATION VISUALIZATIONS        ")
print("===================================================\n")

key_figures = [
    ('Multi-Version Comparison (v1 vs v1.1 vs v1.2)', 'figures/v1.2/version_comparison_all.png'),
    ('v1.2 Model Performance Comparison', 'figures/v1.2/model_comparison_v1.2.png'),
    ('Per-Class F1 Score Heatmap by Position', 'figures/v1.2/per_class_f1_heatmap_v1.2.png'),
    ('Random Forest Top 25 Feature Importances', 'figures/v1.2/feature_importance_rf_v1.2.png'),
    ('Physical Attributes (Height & Weight) by Position', 'figures/v1.2/height_weight_by_position_v1.2.png'),
]

for title, path in key_figures:
    if os.path.exists(path):
        print(f"\n{'='*60}")
        print(f"📊 {title}")
        print(f"   Saved at: {path}")
        print(f"{'='*60}")
        display(Image(filename=path, width=850))

print("\n===================================================")
print("              CONFUSION MATRICES                   ")
print("===================================================\n")

cm_figures = sorted(glob.glob('figures/v1.2/*confusion_matrix*.png'))
for cm_path in cm_figures:
    print(f"\n📉 {os.path.basename(cm_path)}")
    display(Image(filename=cm_path, width=700))


## 7. Summary & Saved Google Drive Artifacts
Review the tuned hyperparameters recorded in `configs/v1.2.json` and verify all model binaries saved in your Google Drive.


In [ ]:
import json

print("=== Tuned Best Parameters (configs/v1.2.json) ===")
if os.path.exists('configs/v1.2.json'):
    with open('configs/v1.2.json', 'r') as f:
        cfg = json.load(f)
    print(json.dumps(cfg.get('best_params', {}), indent=2))

print("\n=== Trained Model Artifacts (models/v1.2/) ===")
if os.path.exists('models/v1.2'):
    for m in sorted(os.listdir('models/v1.2')):
        size_mb = os.path.getsize(os.path.join('models/v1.2', m)) / (1024 * 1024)
        print(f"  ✔ {m} ({size_mb:.2f} MB)")
